# RUN on BASE kernel!

## Meshing based on already segmented mask

- original surface mesh code from https://forum.image.sc/t/converting-3d-binary-segmentation-to-mesh/106252/6
- 3d tetrahedral mesh
    - input: mask 0/1
    - output: vtk mesh
    - included: smoothing + cleaning 

In [1]:
import numpy as np
import numpy
import os

import matplotlib.pyplot as plt
# import skimage
import vtk
from vtkmodules.numpy_interface import dataset_adapter as dsa
from vtk.util import numpy_support


base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/"

folder = "VTI_shared_view/segmentations/"

mask = numpy.load(base + folder + "labels_mask.npy")

export_name = base + folder + "mesh"

In [2]:
def getTetraMesh(stack, filename):
    """
    Generates tetrahedral volume mesh from a 3D binary mask.
    
    stack: 3D numpy array (z,y,x or any ordering — assumed consistent)
    """

    d0, d1, d2 = stack.shape

    # ---------------------------
    # Create vtkImageData
    # ---------------------------
    img = vtk.vtkImageData()
    img.SetDimensions(d0, d1, d2)
    img.SetSpacing(1.0, 1.0, 1.0)
    img.SetOrigin(0.0, 0.0, 0.0)

    # Fast scalar assignment (correct ordering)
    flat = stack.ravel(order='F')  # VTK expects Fortran order
    vtk_data = numpy_support.numpy_to_vtk(
        flat.astype(np.uint8), deep=True, array_type=vtk.VTK_UNSIGNED_CHAR
    )
    img.GetPointData().SetScalars(vtk_data)

    # ---------------------------
    # Extract surface
    # ---------------------------
    snets = vtk.vtkSurfaceNets3D()
    snets.SetInputData(img)
    snets.SetOutputMeshTypeToTriangles()
    snets.Update()

    surface = snets.GetOutput()


    # ---------------------------
    # Smoothen surface (optional)
    # ---------------------------

    smoother = vtk.vtkSmoothPolyDataFilter()
    smoother.SetInputData(surface)
    smoother.SetNumberOfIterations(30)
    smoother.SetRelaxationFactor(0.1)
    smoother.FeatureEdgeSmoothingOff()
    smoother.BoundarySmoothingOn()
    smoother.Update()

    # ---------------------------
    # Clean surface 
    # ---------------------------
    cleaner = vtk.vtkCleanPolyData()

    # if not smoothing
    # cleaner.SetInputData(surface)
    cleaner.SetInputData(smoother.GetOutput())

    cleaner.Update()

    # ---------------------------
    # Generate tetrahedral mesh
    # ---------------------------
    delaunay = vtk.vtkDelaunay3D()
    delaunay.SetInputData(cleaner.GetOutput())
    delaunay.SetTolerance(0.01)
    delaunay.Update()

    tetra_mesh = delaunay.GetOutput()

    # ---------------------------
    # Write to VTU (recommended)
    # ---------------------------
    writer = vtk.vtkXMLUnstructuredGridWriter()
    writer.SetFileName(filename)
    writer.SetInputData(tetra_mesh)
    writer.Write()

    return tetra_mesh



def getSurfaceMesh(stack, filename,
                   smoothing_iterations=20,
                   pass_band=0.1):
    """
    Generate smoothed surface mesh from 3D binary mask
    and export as STL for Gmsh.
    """

    d0, d1, d2 = stack.shape

    # -------------------------
    # Create vtkImageData
    # -------------------------
    img = vtk.vtkImageData()
    img.SetDimensions(d0, d1, d2)
    img.SetSpacing(1.0, 1.0, 1.0)
    img.SetOrigin(0.0, 0.0, 0.0)

    # Fast and correct scalar assignment (VTK uses Fortran order)
    flat = stack.ravel(order='F')
    vtk_data = numpy_support.numpy_to_vtk(
        flat.astype(np.uint8),
        deep=True,
        array_type=vtk.VTK_UNSIGNED_CHAR
    )
    img.GetPointData().SetScalars(vtk_data)

    # -------------------------
    # Surface extraction
    # -------------------------
    snets = vtk.vtkSurfaceNets3D()
    snets.SetInputData(img)
    snets.SetOutputMeshTypeToTriangles()
    snets.Update()

    surface = snets.GetOutput()

    # Clean duplicate points
    cleaner = vtk.vtkCleanPolyData()
    cleaner.SetInputData(surface)
    cleaner.Update()

    # Triangles only
    triFilter = vtk.vtkTriangleFilter()
    triFilter.SetInputData(cleaner.GetOutput())
    triFilter.Update()

    # Normals
    normals = vtk.vtkPolyDataNormals()
    normals.SetInputData(triFilter.GetOutput())
    normals.ConsistencyOn()
    normals.AutoOrientNormalsOn()
    normals.SplittingOff()
    normals.Update()

    # Fill holes
    holeFiller = vtk.vtkFillHolesFilter()
    holeFiller.SetInputData(normals.GetOutput())
    holeFiller.SetHoleSize(1e6)  # large enough to close all small gaps
    holeFiller.Update()

    # Keep only largest connected component
    connectivity = vtk.vtkConnectivityFilter()
    connectivity.SetInputData(holeFiller.GetOutput())
    connectivity.SetExtractionModeToLargestRegion()
    connectivity.Update()

    repaired_surface = connectivity.GetOutput()

    # Check non-manifold edges
    featureEdges = vtk.vtkFeatureEdges()
    featureEdges.SetInputData(repaired_surface)
    featureEdges.NonManifoldEdgesOn()
    featureEdges.BoundaryEdgesOn()
    featureEdges.FeatureEdgesOff()
    featureEdges.ManifoldEdgesOff()
    featureEdges.Update()
    print("Problem edges:", featureEdges.GetOutput().GetNumberOfCells())

    stlWriter = vtk.vtkSTLWriter()
    stlWriter.SetInputData(repaired_surface)
    stlWriter.SetFileName(filename)
    stlWriter.Write()


    # mesh_tetra = getTetraMesh(mask, export_name +"_tetra.vtu")
mesh_surf = getSurfaceMesh(mask, export_name +"_repaired_surface.stl")




Problem edges: 0


2026-02-16 16:28:01.315 (   0.314s) [          AED649]   vtkSurfaceNets3D.cxx:2322  INFO| Executing Surface Nets 3D
2026-02-16 16:28:01.321 (   0.320s) [          AED649]   vtkSurfaceNets3D.cxx:2396  INFO| Extracted: 15996 points, 15994 quads
2026-02-16 16:28:01.321 (   0.320s) [          AED649]   vtkSurfaceNets3D.cxx:1816  INFO| Smoothing output
2026-02-16 16:28:01.321 (   0.320s) [          AED649]vtkConstrainedSmoothing:448   INFO| Executing constrained smoothing filter
2026-02-16 16:28:01.323 (   0.322s) [          AED649]   vtkSurfaceNets3D.cxx:2032  INFO| Transforming output mesh type to: 1
2026-02-16 16:28:01.323 (   0.322s) [          AED649]   vtkSurfaceNets3D.cxx:2447  INFO| Triangulated to produce: 31988 triangles
